# Arm C — Data-Driven Clusters (k-Prototypes)

This notebook implements Arm C of the study: patients are partitioned into data-driven clusters using k-prototypes, and a classifier is trained independently within each cluster. Arm C addresses RQ2 by comparing cluster-stratified modelling against the non-stratified baseline established in Arm A (global_model_armA.ipynb), and RQ3 by comparing it against the sex-stratified subgroups of Arm B (clinical_stratified_armB.ipynb).

k-prototypes is used because the feature set contains both continuous and categorical clinical variables; it combines Euclidean distance on continuous features with simple-matching distance on categorical features, unlike k-means (continuous only) or k-modes (categorical only) (proposal, Section 3.3).

Arm C reuses Arm A's dataset, feature definitions, classifiers, hyperparameter grids, nested cross-validation procedure, evaluation metrics, and outer fold partitions unchanged. The only methodological difference from Arm A is that patients are first grouped into clusters — fitted independently within each outer training fold, never on validation data — and a separate logistic regression and random forest are fitted within each cluster. Predictions from all cluster models are pooled back into a single population-level validation set for the primary comparison against Arm A and Arm B, as required by the proposal (Section 3.5).

**Hypothesis derived from the Arm B analysis (2026-09-09, written before running this notebook).** This prediction was drafted during an AI-assisted planning conversation that had already seen Arm B's results, then reviewed and recorded here before the notebook was run -- it is a hypothesis extrapolated from Arm B's finding, not an independently-formed prior belief arrived at before any of this study's own results existed. It is left in place rather than deleted or edited after the fact because it was genuinely written before this notebook ran and the date is real; see the Outcome cell directly below for what actually happened. Arm A outperformed Arm B by roughly 5.8 accuracy points for logistic regression (0.842 vs 0.784), a gap that traced almost entirely to Arm B halving the training data available to each subgroup model rather than to any absent sex-based interaction signal: the female subgroup's F1 was unstable (0.461 ± 0.428 across folds; F1 hit 0.000 in two of five folds), which Arm B's own Section 14 traces to a regularisation-and-threshold interaction -- the inner grid search selected the strongest L2 penalty on offer in exactly those two folds, which flattened predicted probabilities below the fixed 0.5 threshold even though the model still ranked patients correctly (female ROC-AUC stayed at 0.80-0.98 throughout) -- rather than to class imbalance or a small-sample learning failure. Arm C's clusters will be smaller than Arm B's 96-patient female subgroup — on 297 patients, k ∈ {2, 3, 4} gives clusters averaging 148, 99, or 74 patients before any imbalance, and k-prototypes has no reason to split evenly — and clusters have no equivalent to sex's ~30-point base-rate separation (55.7% vs 26.0% disease prevalence) to justify the split. **Prediction: Arm C will underperform both Arm A and Arm B on pooled accuracy, F1, and ROC-AUC**, for the same sample-size reason that hurt Arm B, plausibly compounded further. If this prediction holds, it is evidence that the mechanism identified in Arm B — variance cost of stratification exceeding its bias benefit at n≈300 — generalises across stratification strategies rather than being specific to sex. If it does not hold, that is itself a result worth investigating rather than a failure of the prediction; see Section 15.

**Outcome (2026-09-09, recorded after running this notebook and `cross_arm_comparison.ipynb`).** The prediction's *direction* was wrong on the B-vs-C leg: Arm C did not underperform Arm B on every metric as predicted. On pooled accuracy, Arm C beat Arm B for logistic regression (0.814 vs 0.784) and matched it almost exactly for random forest (0.801 vs 0.801); on F1, Arm C beat Arm B for logreg (0.792 vs 0.753, winning 4 of 5 folds) and was essentially tied for rf. On the A-vs-C leg the prediction's direction held -- Arm C trails Arm A on every metric for both classifiers -- but by a narrower margin than Arm B trails Arm A by.

Per `cross_arm_comparison.ipynb` (Task 3), however, **none of Arm C's arm-to-arm accuracy, F1, or ROC-AUC gaps are statistically distinguishable from fold-to-fold noise**: A-vs-C accuracy p=0.307 (logreg) / 0.435 (rf); B-vs-C accuracy p=0.208 (logreg) / 0.996 (rf), all by paired t-test (Wilcoxon agrees), and those p-values are themselves anti-conservative (`cross_arm_comparison.ipynb` Section 5). So the honest reading is neither "the prediction was right" nor "the prediction was wrong" but: **the specific direction predicted for B-vs-C did not hold in this run's point estimate, but neither direction is distinguishable from noise at 5 folds, so this is not strong evidence against the underlying mechanism either.**

What the underlying mechanism claimed -- that stratification does not reliably help at n≈300, because the variance cost of splitting the data outweighs any bias benefit from a better-tailored per-group model -- is still supported: neither Arm B's nor Arm C's differences from Arm A are statistically distinguishable. Arm C's edge over Arm B in this run's point estimate traces to a specific, identifiable cause rather than to stratification "working" after all: Task 2's scaling correction changed what k-prototypes finds, and the corrected clusters turned out to track the disease-severity gradient closely (Section 15) -- a structure the classifier was already positioned to learn from the raw features, not a new source of information stratification uniquely unlocked. The two cluster-specific F1 standard deviations (0.44 and 0.41, Section 15) show the same regularisation-and-threshold instability documented for Arm B's female subgroup (Task 4, Arm B Section 14) is present in Arm C too, just distributed across both clusters instead of concentrated in one subgroup -- consistent with a shared mechanism, not a refutation of one.

## 2. Imports and configuration

`RANDOM_STATE`, `K_OUTER`, and `K_INNER` match Arm A and Arm B exactly, for the same reasons documented there. Three additional settings are specific to Arm C's clustering step:

- `K_CLUSTER_CANDIDATES = [2, 3, 4]`: the candidate cluster counts considered within each outer training fold, matching the proposal's requirement to keep the number of clusters small given the dataset size (Section 3.3).
- `MIN_CLUSTER_SIZE = 40`: the minimum number of training patients a cluster must contain before a classifier is fitted within it. This is smaller than Arm B's smallest subgroup (~80 female training patients per fold) because Arm C's clusters are expected to be smaller than Arm B's subgroups; the guard in Section 10 also requires enough patients of each class for 5-fold inner cross-validation to be well-defined, which is a separate and stricter condition than the raw size threshold.
- `N_INIT_KPROTO = 10`: the number of random centroid initialisations k-prototypes runs per candidate k, matching the library default so results are typical rather than best-of-many.

In [1]:
import os
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, silhouette_score, adjusted_rand_score

from kmodes.kprototypes import KPrototypes

# Configuration (RANDOM_STATE, K_OUTER, K_INNER must match Arm A and Arm B)
RANDOM_STATE = 42
K_OUTER      = 5
K_INNER      = 5

# Arm C-specific clustering configuration
K_CLUSTER_CANDIDATES = [2, 3, 4]
MIN_CLUSTER_SIZE      = 40
N_INIT_KPROTO         = 10

# Path constants -- this notebook lives in notebooks/, so PROJECT_ROOT is
# one level up; all data and outputs are read/written relative to it.
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "heart+disease")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
PREDICTIONS_DIR = os.path.join(RESULTS_DIR, "predictions")
DIAGNOSTICS_DIR = os.path.join(PROJECT_ROOT, "diagnostics")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)

## 3. Load dataset

Arm C uses the same cleaned Cleveland extract as Arm A and Arm B. The cleaning procedure is documented in `data_cleaning.ipynb` and is not repeated here.

In [2]:
df = pd.read_csv(os.path.join(DATA_DIR, "cleveland_clean.csv"))
print("Loaded shape:", df.shape)
assert df.shape[0] == 297, "Arm C expects the same 297-record cleaned dataset used in Arm A and Arm B."
df.head()

Loaded shape: (297, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


## 4. Target construction

The original Cleveland target, `num`, represents the presence/severity of heart disease. For binary classification, observations with `num = 0` are coded as class 0 (absence of disease), while observations with `num > 0` are coded as class 1 (presence of disease). The derived `check` column is excluded from the feature matrix together with `num`, for the same leakage reason documented in Arm A.

In [3]:
target_col = "num"

y = (df[target_col] > 0).astype(int)
X = df.drop(columns=[target_col, "check"], errors="ignore")

print("Samples:", len(df), " Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297  Features: 13
Class balance:
num
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 5. Feature definition

Two different groupings of the same 13 features are used in this notebook, for two different purposes:

- **Classifier feature grouping** — identical to Arm A and Arm B: continuous features are standardised, nominal category codes are one-hot encoded, and binary/count features pass through unchanged. This grouping is used inside every `Pipeline` that fits a logistic regression or random forest, exactly as in Arm A and Arm B, so the classifiers themselves see the same feature representation in all three arms.
- **Clustering feature grouping** — used only by k-prototypes, which requires features split into a numeric block (compared by Euclidean distance) and a categorical block (compared by simple matching), rather than one-hot encoded. `cp`, `restecg`, `slope`, and `thal` — the same four features one-hot encoded for the classifiers — are passed to k-prototypes as raw categorical codes: one-hot encoding them first would turn each category into several near-binary numeric columns under Euclidean distance, which is exactly what using k-prototypes over k-means was meant to avoid. `age`, `trestbps`, `chol`, `thalach`, and `oldpeak` are standardised and passed as numeric; `sex`, `fbs`, `exang`, and `ca` are *also* standardised for clustering, alongside the continuous block, in one `StandardScaler` (Section 6 explains why -- leaving them on their raw 0/1-3 scale silently down-weights them under squared Euclidean distance). This is specific to the clustering step: for the classifiers, `sex`, `fbs`, `exang`, and `ca` remain unscaled passthrough columns, exactly as in Arm A, since standardising a binary column has no effect on a tree split and only rescales (not reweights) a logistic regression coefficient.

In [4]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]
passthrough = ["sex", "fbs", "exang", "ca"]

# Clustering feature order: numeric block (continuous + passthrough) then
# categorical block (nominal). Categorical column indices are positional
# within this fixed order, not within the original dataframe.
cluster_numeric_cols     = continuous + passthrough
cluster_categorical_cols = nominal
cluster_feature_order    = cluster_numeric_cols + cluster_categorical_cols
cluster_categorical_idx  = list(range(len(cluster_numeric_cols), len(cluster_feature_order)))

print("Continuous:", continuous)
print("Nominal (one-hot for classifiers / categorical for clustering):", nominal)
print("Passthrough (already binary/count):", passthrough)
print("Clustering categorical column indices:", cluster_categorical_idx)

Continuous: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Nominal (one-hot for classifiers / categorical for clustering): ['cp', 'restecg', 'slope', 'thal']
Passthrough (already binary/count): ['sex', 'fbs', 'exang', 'ca']
Clustering categorical column indices: [9, 10, 11, 12]


## 6. Data-driven cluster exploratory summary (descriptive only — not used for evaluation)

Before nested cross-validation, k-prototypes is fitted on the **entire** dataset purely to characterise what the algorithm finds, in the same spirit as Arm B's Section 6 sex-based summary. This whole-dataset fit is descriptive only: it plays no role in the leakage-safe evaluation in Sections 10-14, where clustering is refit independently within each outer training fold. The helper functions defined here (`build_cluster_arrays`, `mixed_distance_matrix`, `fit_kprototypes_select_k`, `predict_clusters`) are reused unchanged by the per-fold procedure in Section 10.

**A scaling correction.** The numeric block passed to k-prototypes contains the five continuous features and the four passthrough features (`sex`, `fbs`, `exang`, `ca`). If only the continuous features are standardised, the passthrough features enter as raw 0/1 or 0-3 values with roughly a fifth of the variance of a standardised feature; since squared Euclidean distance weights every numeric feature by its scale, this silently down-weights the passthrough block by about the same factor -- an unintended feature weighting inside the step this entire arm depends on. **All numeric features are therefore standardised together with one `StandardScaler`**; `nominal` stays as raw categorical codes passed via `categorical=`, which is the actual reason for using k-prototypes over k-means and does not change. Because this choice visibly changes the resulting clusters, a sensitivity analysis further below re-runs the whole-dataset clustering both ways and reports whether the RQ3 conclusion holds under both.

In [5]:
def build_cluster_arrays(frame, scaler=None, fit_scaler=False, scale_passthrough=True):
    """Build the (numeric, categorical) arrays k-prototypes expects from a
    dataframe slice, in the fixed column order defined in Section 5.

    `scale_passthrough=True` (the corrected, default behaviour used
    everywhere except the sensitivity analysis below) standardises the
    passthrough features (sex/fbs/exang/ca) together with the continuous
    block in one StandardScaler, so squared Euclidean distance does not
    silently down-weight them. `scale_passthrough=False` reproduces the
    original, uncorrected behaviour (passthrough left on its raw 0/1-3
    scale) and exists only so the sensitivity analysis can show the
    difference it makes."""
    numeric_cols = continuous + passthrough if scale_passthrough else continuous
    if fit_scaler:
        scaler = StandardScaler().fit(frame[numeric_cols])
    scaled = scaler.transform(frame[numeric_cols])
    if scale_passthrough:
        Xnum = scaled
    else:
        Xnum = np.hstack([scaled, frame[passthrough].to_numpy(dtype=float)])
    Xcat = frame[nominal].to_numpy()
    return Xnum, Xcat, scaler


def mixed_distance_matrix(Xnum_a, Xcat_a, Xnum_b, Xcat_b, gamma):
    """Pairwise dissimilarity between two point sets, using exactly the
    k-prototypes cost function (squared Euclidean on the numeric block plus
    gamma times simple-matching count on the categorical block), so the
    silhouette score used to select k is consistent with what k-prototypes
    itself minimises. Squared Euclidean is not a true metric (it fails the
    triangle inequality), so silhouette values computed from this distance
    are internally consistent for comparing candidate k's here, but are not
    numerically comparable to silhouette scores reported elsewhere in the
    literature, which are almost always computed on true Euclidean or
    Gower distance."""
    num_sq = ((Xnum_a[:, None, :] - Xnum_b[None, :, :]) ** 2).sum(axis=2)
    cat_mismatch = (Xcat_a[:, None, :] != Xcat_b[None, :, :]).sum(axis=2)
    return num_sq + gamma * cat_mismatch


def fit_kprototypes_select_k(frame, k_candidates, min_cluster_size, random_state, n_init, scale_passthrough=True):
    """Fit k-prototypes on `frame`'s patients only, for each candidate k,
    using predictor variables alone. Outcome labels are never passed to
    this function, so they cannot influence which k is selected -- this is
    now structurally, not just procedurally, true.

    For each candidate k:
    1. SIZE GUARD (label-blind) -- reject k if any resulting cluster has
       fewer than `min_cluster_size` patients.
    2. SELECTION (unsupervised) -- among the k's that pass, the one with
       the highest silhouette score under the k-prototypes mixed distance
       is chosen.

    A separate check on whether each selected cluster has enough patients
    of each outcome class to support 5-fold inner CV happens later, after
    k is already fixed, in `cluster_or_fallback` (Section 10) -- it never
    feeds back into this function or changes k.

    Returns (model, scaler, labels, k, silhouette), or
    (None, None, None, None, None) if no candidate k is feasible."""
    Xnum, Xcat, scaler = build_cluster_arrays(frame, fit_scaler=True, scale_passthrough=scale_passthrough)

    feasible = []
    for k in k_candidates:
        if len(frame) < min_cluster_size * k:
            continue  # cannot satisfy the size guard for this k regardless of split

        model = KPrototypes(n_clusters=k, init="Cao", n_init=n_init,
                             random_state=random_state, verbose=0)
        labels = model.fit_predict(np.hstack([Xnum, Xcat]), categorical=cluster_categorical_idx)

        sizes = np.bincount(labels, minlength=k)
        if sizes.min() < min_cluster_size:
            continue

        feasible.append((k, model, labels))

    # --- selection criterion (unsupervised: silhouette only) ---
    best = None
    for k, model, labels in feasible:
        dist = mixed_distance_matrix(Xnum, Xcat, Xnum, Xcat, model.gamma)
        sil = silhouette_score(dist, labels, metric="precomputed")
        if best is None or sil > best["silhouette"]:
            best = {"model": model, "scaler": scaler, "labels": labels, "k": k, "silhouette": sil}

    if best is None:
        return None, None, None, None, None
    return best["model"], best["scaler"], best["labels"], best["k"], best["silhouette"]


def predict_clusters(model, scaler, frame, scale_passthrough=True):
    """Assign patients in `frame` to the nearest already-fitted centroid.
    Never refits k-prototypes; used for validation-fold patients only.
    `scale_passthrough` must match what the model was fit with."""
    Xnum, Xcat, _ = build_cluster_arrays(frame, scaler=scaler, fit_scaler=False, scale_passthrough=scale_passthrough)
    return model.predict(np.hstack([Xnum, Xcat]), categorical=cluster_categorical_idx)

In [6]:
# Descriptive only: fit once on all 297 patients, with the corrected scaling
# (scale_passthrough=True, the default), to characterise what k-prototypes
# finds. Independent of the leakage-safe per-fold procedure in Sections 10-14.
desc_model, desc_scaler, desc_labels, desc_k, desc_sil = fit_kprototypes_select_k(
    X, K_CLUSTER_CANDIDATES, MIN_CLUSTER_SIZE, RANDOM_STATE, N_INIT_KPROTO
)

if desc_model is None:
    print("No candidate k in", K_CLUSTER_CANDIDATES, "satisfied the size guard "
          "on the whole dataset; no descriptive clustering to report.")
else:
    print(f"Whole-dataset descriptive clustering selected k={desc_k} (silhouette={desc_sil:.3f})")

    desc = pd.DataFrame({
        "cluster": desc_labels,
        "sex": df["sex"].map({1.0: "male", 0.0: "female"}),
        "disease": y,
    })
    desc_summary = desc.groupby("cluster").agg(
        n=("disease", "size"),
        disease_rate=("disease", "mean"),
        pct_male=("sex", lambda s: (s == "male").mean()),
    ).round(3)
    desc_summary["pct_male"] = (desc_summary["pct_male"] * 100).round(1)
    print(desc_summary)
    desc_summary.to_csv(os.path.join(DIAGNOSTICS_DIR, "armC_cluster_descriptive_summary.csv"))
    print("Saved armC_cluster_descriptive_summary.csv")

    print()
    print("Cluster membership vs sex:")
    desc_crosstab = pd.crosstab(desc_labels, desc["sex"], margins=True)
    print(desc_crosstab)
    desc_crosstab.to_csv(os.path.join(RESULTS_DIR, "armC_cluster_vs_sex_crosstab.csv"))
    print("Saved armC_cluster_vs_sex_crosstab.csv")

    ari_outcome = adjusted_rand_score(y, desc_labels)
    ari_sex = adjusted_rand_score(df["sex"], desc_labels)
    print(f"Adjusted Rand index vs outcome: {ari_outcome:.3f}")
    print(f"Adjusted Rand index vs sex:     {ari_sex:.3f}")

Whole-dataset descriptive clustering selected k=3 (silhouette=0.324)
           n  disease_rate  pct_male
cluster                             
0        103         0.825      77.7
1        151         0.212      59.6
2         43         0.465      72.1
Saved armC_cluster_descriptive_summary.csv

Cluster membership vs sex:
sex    female  male  All
row_0                   
0          23    80  103
1          61    90  151
2          12    31   43
All        96   201  297
Saved armC_cluster_vs_sex_crosstab.csv
Adjusted Rand index vs outcome: 0.266
Adjusted Rand index vs sex:     -0.002


In [7]:
# Sensitivity analysis (Task 2): does the scaling correction change the
# headline RQ3 finding? Re-run the whole-dataset descriptive clustering
# both ways and compare cluster sizes, disease rate, %male, and ARI against
# both outcome and sex.
sensitivity_rows = []
for label, scale_pt in [("passthrough unscaled (uncorrected)", False), ("passthrough standardised (corrected)", True)]:
    model_s, scaler_s, labels_s, k_s, sil_s = fit_kprototypes_select_k(
        X, K_CLUSTER_CANDIDATES, MIN_CLUSTER_SIZE, RANDOM_STATE, N_INIT_KPROTO,
        scale_passthrough=scale_pt,
    )
    if model_s is None:
        sensitivity_rows.append({"scaling": label, "k": None})
        continue
    sizes_s = np.bincount(labels_s, minlength=k_s)
    disease_rates = [round(y[labels_s == c].mean(), 3) for c in range(k_s)]
    pct_males = [round((df["sex"][labels_s == c] == 1.0).mean() * 100, 1) for c in range(k_s)]
    sensitivity_rows.append({
        "scaling": label,
        "k": k_s,
        "cluster_sizes": sizes_s.tolist(),
        "disease_rates": disease_rates,
        "pct_male": pct_males,
        "ari_vs_outcome": round(adjusted_rand_score(y, labels_s), 3),
        "ari_vs_sex": round(adjusted_rand_score(df["sex"], labels_s), 3),
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(os.path.join(DIAGNOSTICS_DIR, "armC_scaling_sensitivity.csv"), index=False)
print("Saved armC_scaling_sensitivity.csv")
sensitivity_df

Saved armC_scaling_sensitivity.csv


,scaling,k,cluster_sizes,disease_rates,pct_male,ari_vs_outcome,ari_vs_sex
0,passthrough unscaled (uncorrected),2,"[159, 138]","[0.226, 0.732]","[66.7, 68.8]",0.256,-0.004
1,passthrough standardised (corrected),3,"[103, 151, 43]","[0.825, 0.212, 0.465]","[77.7, 59.6, 72.1]",0.266,-0.002


**Conclusion.** The RQ3 finding -- that k-prototypes clusters have no relationship to sex -- holds under both scalings: adjusted Rand index vs sex is essentially zero either way (-0.004 unscaled, -0.002 standardised). What the correction *does* change is the cluster count and composition the whole-dataset fit prefers: k=2 (sizes 159/138, disease rates 22.6%/73.2%) under the uncorrected scaling, versus k=3 (sizes 103/151/43, disease rates 82.5%/21.2%/46.5%) once the passthrough block is standardised -- adjusted Rand index vs outcome also rises slightly, from 0.256 to 0.266. So the *headline RQ3 finding is robust to this modelling choice*, but the specific shape of the clusters used for Section 15's characterization is not; everywhere else in this notebook, the corrected (standardised) scaling is the default and is what actually drives the leakage-safe evaluation in Sections 10-14.

In [8]:
# Silhouette by k (Task 6b): a structure diagnostic, independent of the
# feasibility guard in Section 10 -- computed here for every k from 2 to 6
# on the whole dataset, using the corrected scaling, purely to see how
# strongly the data support any particular number of clusters.
Xnum_full, Xcat_full, _ = build_cluster_arrays(X, fit_scaler=True)
silhouette_by_k = {}
for k in range(2, 7):
    model_k = KPrototypes(n_clusters=k, init="Cao", n_init=N_INIT_KPROTO, random_state=RANDOM_STATE, verbose=0)
    labels_k = model_k.fit_predict(np.hstack([Xnum_full, Xcat_full]), categorical=cluster_categorical_idx)
    dist_k = mixed_distance_matrix(Xnum_full, Xcat_full, Xnum_full, Xcat_full, model_k.gamma)
    silhouette_by_k[k] = silhouette_score(dist_k, labels_k, metric="precomputed")

silhouette_by_k_df = pd.Series(silhouette_by_k, name="silhouette").rename_axis("k").reset_index()
silhouette_by_k_df.to_csv(os.path.join(DIAGNOSTICS_DIR, "armC_silhouette_by_k.csv"), index=False)
print("Saved armC_silhouette_by_k.csv")
silhouette_by_k_df.round(3)

Saved armC_silhouette_by_k.csv


,k,silhouette
0,2,0.298
1,3,0.324
2,4,0.285
3,5,0.257
4,6,0.261


Silhouette is known to be biased toward small k on many real datasets, so the data-driven k-selection in Sections 6 and 10 is doing less independent work than it might appear -- k=2 tends to win partly because small k tends to win regardless of the underlying structure, not only because 2 is a uniquely good fit here. The silhouette values above are modest by the conventional rule of thumb (below roughly 0.25 indicates weak or no real structure, 0.25-0.5 a weak-to-reasonable structure, above 0.5 a strong one): they indicate a real but not strongly separated grouping, not a small number of tight, well-isolated clusters. Combined with the note in `mixed_distance_matrix`'s docstring -- squared Euclidean is not a true metric, so these values are not comparable to silhouette scores reported elsewhere in the literature -- the right reading of "k=2, silhouette ≈0.3" is "weak-to-moderate structure, and the method's preference for few clusters should not be over-interpreted as strong evidence for exactly two subgroups."

## 7. Preprocessing

Preprocessing that estimates parameters from the data is kept inside a scikit-learn `Pipeline` together with each classifier, exactly as in Arm A and Arm B. Because a separate pipeline is fitted per outer fold per cluster (Section 11), the scaler and encoder only ever see the training portion of that cluster. The `ColumnTransformer` definition is identical to Arm A's and Arm B's — this is the classifier feature grouping from Section 5, not the clustering feature grouping.

In [9]:
# Fitted only inside the pipeline, on training-fold data (see Section 11), never on the full dataset.
preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                       continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

## 8. Reuse Arm A's outer folds

The proposal requires identical outer cross-validation fold partitions across all arms (Section 3.5). Arm A already created and saved this partition to `fold_id.csv`; Arm C loads it directly rather than generating a new one, so every patient sits in exactly the same outer fold in Arm A, Arm B, and Arm C. An MD5 hash of `fold_id.csv`, written by Arm A into `run_manifest.json`, is checked here too (matching Arm B's Section 8), so a deleted, regenerated, or otherwise stale fold file is caught immediately.

In [10]:
FOLD_FILE = os.path.join(PROJECT_ROOT, "fold_id.csv")
if not os.path.exists(FOLD_FILE):
    raise FileNotFoundError(
        f"{FOLD_FILE} not found. Arm C requires the outer fold partition created by "
        "global_model_armA.ipynb; run that notebook first."
    )

fold_id = pd.read_csv(FOLD_FILE)["fold"].to_numpy()
if len(fold_id) != len(df):
    raise ValueError(
        f"{FOLD_FILE} has {len(fold_id)} entries but the current dataset has {len(df)} rows."
    )
print(f"Loaded outer fold assignment from {FOLD_FILE} (shared with Arm A and Arm B).")

# Fold-file integrity guard (Task 9 / Arm A Section 8, Arm B Section 8): fail
# loudly if this fold_id.csv is not the one Arm A actually produced.
MANIFEST_FILE = os.path.join(PROJECT_ROOT, "run_manifest.json")
if not os.path.exists(MANIFEST_FILE):
    raise FileNotFoundError(
        f"{MANIFEST_FILE} not found. Arm C requires the manifest written by "
        "global_model_armA.ipynb; run that notebook first."
    )
with open(MANIFEST_FILE) as f:
    manifest = json.load(f)
fold_id_md5 = hashlib.md5(open(FOLD_FILE, "rb").read()).hexdigest()
assert fold_id_md5 == manifest["fold_id_md5"], (
    f"{FOLD_FILE} MD5 ({fold_id_md5}) does not match run_manifest.json "
    f"({manifest['fold_id_md5']}); it was regenerated or edited since Arm A ran. "
    "Re-run global_model_armA.ipynb and then this notebook, in that order."
)
print(f"fold_id.csv MD5 verified against run_manifest.json: {fold_id_md5}")

print("Fold sizes:", np.bincount(fold_id))
print("Positives per fold:", np.bincount(fold_id[y.values == 1]))

Loaded outer fold assignment from /Users/faye/Desktop/FIT2082/Research_heartDisease/FIT2082_Research/fold_id.csv (shared with Arm A and Arm B).
fold_id.csv MD5 verified against run_manifest.json: 4d54a69c577f498114826fb495d7eaa8
Fold sizes: [60 60 59 59 59]
Positives per fold: [28 28 27 27 27]


## 9. Model definitions and hyperparameter grids

Exactly the same two classifiers and hyperparameter grids as Arm A and Arm B. Cluster models are tuned from the same candidate hyperparameters and the same selection procedure as the global model; only the data used to fit them, and how that data is grouped, differs.

In [11]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}

## 10. Per-fold clustering procedure and guards

Within each outer fold, clustering is fit using only that fold's training patients, with the same helper functions defined in Section 6 (`fit_kprototypes_select_k`, `predict_clusters`): the scaler for the continuous block and the k-prototypes model are both fit on training-fold patients only, and validation patients are assigned to the nearest already-fitted centroid via `.predict()` — never by refitting k-prototypes on data that includes them. This is the same leakage-prevention pattern Arm A and Arm B use for preprocessing, applied to an additional step: cluster formation itself.

Two stages, decided before running this notebook and applied identically to every fold, kept structurally separate rather than combined into one check:

- **Stage 1 — k selection (label-blind).** `fit_kprototypes_select_k` no longer accepts outcome labels as an argument at all, so they cannot influence which k is chosen, even in principle. A candidate k is rejected if any resulting training cluster has fewer than `MIN_CLUSTER_SIZE` (40) patients (a size guard on predictor-space clusters alone); among the k's that pass, the one with the highest silhouette score (computed under the same mixed-distance function k-prototypes itself minimises) is selected. This is the whole of k selection — genuinely unsupervised, not merely "mostly" unsupervised.
- **Stage 2 — feasibility check (classification stage, after k is fixed).** Once k is selected, `cluster_or_fallback` checks whether every resulting training cluster has at least `K_INNER` (5) patients of each outcome class — the condition that actually makes 5-fold stratified inner cross-validation well-defined within that cluster. This check uses training-fold labels only (not a leakage risk) but runs strictly *after* k is chosen and never feeds back into k selection.

**Fallback to the global model.** If Stage 1 finds no feasible k, or Stage 2's check fails for the k that Stage 1 selected, that fold falls back to treating its entire outer-training set as a single cluster — i.e. that fold's contribution to Arm C becomes identical in procedure to Arm A's global model for that fold. This is a pre-specified fallback, not a choice made after seeing which folds are difficult. The two failure modes are recorded separately in `armC_cluster_choice.csv` (`fallback_reason`: `"no_k_passed_size_guard"` vs `"selected_k_failed_class_count"`) rather than collapsed into one flag, and The cell immediately below audits, for every fold and every candidate k, the cluster sizes, whether the size guard passes, and the silhouette score — demonstrating what actually drives k selection now that outcome labels play no role in it.

A separate guard on the **validation** side handles single-class validation clusters: if the validation patients assigned to a cluster happen to contain only one outcome class, `roc_auc_score` is undefined for that cluster and is recorded as `NaN` in the cluster-specific breakdown, reusing the pattern Arm B applied to its sex-specific breakdown. This does not affect the primary pooled metric, which is computed once per fold across all clusters' validation patients combined.

In [12]:
def cluster_or_fallback(X_train_fold, y_train_fold):
    """Select cluster count on predictor variables alone (Fix 1: outcome
    labels never reach `fit_kprototypes_select_k`), then apply a separate,
    pre-declared feasibility check on the classification side: does every
    selected cluster have at least K_INNER training patients of each
    outcome class, the condition that makes 5-fold stratified inner CV
    well-defined? This check runs strictly after k is fixed, uses
    training-fold labels only, never changes k, and never sees validation
    data. If either this check or the label-blind size guard inside
    `fit_kprototypes_select_k` fails, the whole outer-training fold falls
    back to a single cluster (= the global model for that fold) -- the
    same pre-specified fallback either way, but the two failure modes are
    kept distinguishable rather than collapsed into one:

    - "no_k_passed_size_guard": no candidate k in {2, 3, 4} produced every
      cluster >= MIN_CLUSTER_SIZE patients (Fix 1's guard; never looks at y).
    - "selected_k_failed_class_count": a k was selected on predictor
      variables alone, but at least one resulting cluster has fewer than
      K_INNER training patients of one outcome class, discovered only
      after k was already chosen.

    Returns (labels, scaler, model, k_used, fallback, fallback_reason),
    where fallback_reason is None when fallback is False."""
    model, scaler, labels, k, sil = fit_kprototypes_select_k(
        X_train_fold, K_CLUSTER_CANDIDATES, MIN_CLUSTER_SIZE, RANDOM_STATE, N_INIT_KPROTO
    )
    if model is None:
        return np.zeros(len(X_train_fold), dtype=int), None, None, 1, True, "no_k_passed_size_guard"

    y_arr = np.asarray(y_train_fold)
    min_class_counts = [np.bincount(y_arr[labels == c], minlength=2).min() for c in range(k)]
    if min(min_class_counts) < K_INNER:
        return np.zeros(len(X_train_fold), dtype=int), None, None, 1, True, "selected_k_failed_class_count"

    return labels, scaler, model, k, False, None

In [13]:
# Fix 3 audit trail: for every outer fold and every candidate k, show cluster
# sizes, whether the label-blind size guard passes, and the silhouette score
# -- independent of, and prior to, running the main nested CV loop below.
# This makes k selection auditable and demonstrates what actually drives it
# now that outcome labels play no role at all in choosing k.
for k in range(K_OUTER):
    train_audit = (fold_id != k)
    X_train_audit = X[train_audit]
    Xnum_audit, Xcat_audit, _ = build_cluster_arrays(X_train_audit, fit_scaler=True)
    print(f"fold {k}:")
    for cand_k in K_CLUSTER_CANDIDATES:
        model_audit = KPrototypes(n_clusters=cand_k, init="Cao", n_init=N_INIT_KPROTO,
                                   random_state=RANDOM_STATE, verbose=0)
        labels_audit = model_audit.fit_predict(np.hstack([Xnum_audit, Xcat_audit]), categorical=cluster_categorical_idx)
        sizes_audit = np.bincount(labels_audit, minlength=cand_k)
        size_guard_pass = bool(sizes_audit.min() >= MIN_CLUSTER_SIZE)
        dist_audit = mixed_distance_matrix(Xnum_audit, Xcat_audit, Xnum_audit, Xcat_audit, model_audit.gamma)
        sil_audit = silhouette_score(dist_audit, labels_audit, metric="precomputed")
        print(f"    k={cand_k}: sizes={sizes_audit.tolist()}, size_guard_pass={size_guard_pass}, "
              f"silhouette={sil_audit:.3f}")

fold 0:


    k=2: sizes=[147, 90], size_guard_pass=True, silhouette=0.303


    k=3: sizes=[72, 129, 36], size_guard_pass=False, silhouette=0.321


    k=4: sizes=[83, 61, 34, 59], size_guard_pass=False, silhouette=0.266
fold 1:
    k=2: sizes=[141, 96], size_guard_pass=True, silhouette=0.321


    k=3: sizes=[75, 130, 32], size_guard_pass=False, silhouette=0.349


...

    k=2: sizes=[100, 138], size_guard_pass=True, silhouette=0.299


    k=3: sizes=[84, 121, 33], size_guard_pass=False, silhouette=0.327


    k=4: sizes=[31, 67, 84, 56], size_guard_pass=False, silhouette=0.293
fold 4:
    k=2: sizes=[104, 134], size_guard_pass=True, silhouette=0.293


    k=3: sizes=[32, 121, 85], size_guard_pass=False, silhouette=0.330


    k=4: sizes=[64, 54, 88, 32], size_guard_pass=False, silhouette=0.292


## 11. Nested cross-validation within clusters

For each outer fold, clustering is fit once (Section 10) and reused for both classifiers, since cluster membership does not depend on which classifier will later be trained. Unlike Arm B, where the sex partition is fixed by definition and the original loop iterates model-then-fold, this notebook iterates fold-then-model so clustering is not refit identically twice per fold; this is a bookkeeping simplification specific to Arm C and does not change results.

Within each cluster, hyperparameters are selected using `GridSearchCV` with inner stratified 5-fold cross-validation on that cluster's outer-training patients only, exactly as in Arm A and Arm B; the outer validation fold is never seen during hyperparameter selection. Clustering itself is fit once on the whole outer-training set (Section 10) and is **not** refit inside each inner CV split: the inner-validation patients used during hyperparameter search therefore had their features contribute to the cluster boundaries their own model is tuned within. This makes the inner CV score used to pick hyperparameters slightly optimistic, but it does not bias the outer estimate that is this arm's primary result, because outer-validation patients are genuinely untouched -- they enter only via `predict_clusters()` against already-fitted centroids. Refitting k-prototypes inside every inner fold would remove even that slight optimism, but at roughly 5x the clustering cost for a difference that could only ever shift which hyperparameters are chosen, not whether the reported outer performance is leaked; this is a documented decision, not an oversight. ROC-AUC is the tuning criterion, matching Arm A and Arm B. Predicted class labels use the same fixed 0.5 probability threshold as the other two arms.

For every outer fold, every cluster's model predictions on that cluster's validation patients are pooled into a single population-level validation set before computing accuracy, F1, and ROC-AUC — the primary Arm C result, directly comparable to Arm A's and Arm B's pooled metrics. `best_params` is recorded per cluster per fold per model, following Arm B's practice, so that unstable or degenerate cluster models (analogous to Arm B's female-subgroup instability) can be diagnosed after the fact rather than only inferred from aggregate scores.

In [14]:
fold_results = []         # pooled, population-level: primary result
cluster_fold_results = [] # cluster-specific: secondary, descriptive result
cluster_choice = []       # which k (or fallback) each fold picked
best_params_rows = []     # per cluster per fold per model
oof_rows = []
patient_id = df.index.to_numpy()

for k in range(K_OUTER):
    train, validation = (fold_id != k), (fold_id == k)
    X_train_fold, y_train_fold = X[train], y[train]
    X_val_fold = X[validation]

    labels_train, scaler, cluster_model, k_used, fallback, fallback_reason = cluster_or_fallback(X_train_fold, y_train_fold)

    if fallback:
        labels_val = np.zeros(len(X_val_fold), dtype=int)
        print(f"fold {k}: fallback to a single cluster (= global model) for this fold "
              f"[reason: {fallback_reason}].")
    else:
        labels_val = predict_clusters(cluster_model, scaler, X_val_fold)
        print(f"fold {k}: selected k={k_used}, training cluster sizes "
              f"{np.bincount(labels_train, minlength=k_used).tolist()}")

    cluster_choice.append({
        "fold": k, "k_used": k_used, "fallback_to_global": fallback, "fallback_reason": fallback_reason,
        "train_cluster_sizes": np.bincount(labels_train, minlength=k_used).tolist(),
        "val_cluster_sizes": np.bincount(labels_val, minlength=k_used).tolist(),
    })

    for name, (pipe, grid) in models.items():
        pooled_y_true, pooled_proba, pooled_pred = [], [], []
        cluster_aucs, cluster_ns = [], []  # for the sample-size-weighted AUC, Section 13

        for cluster_id in range(k_used):
            cluster_train = train.copy()
            cluster_train[train] = (labels_train == cluster_id)
            cluster_validation = validation.copy()
            cluster_validation[validation] = (labels_val == cluster_id)

            if cluster_validation.sum() == 0:
                continue  # no validation patients were assigned to this cluster this fold

            # Hyperparameter selection uses only this cluster's outer-training
            # patients; the outer validation fold is not seen until scoring below.
            inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
            search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
            search.fit(X[cluster_train], y[cluster_train])

            best = search.best_estimator_
            proba = best.predict_proba(X[cluster_validation])[:, 1]
            pred = (proba >= 0.5).astype(int)  # Fixed threshold, consistent with Arm A and Arm B.
            y_val = y[cluster_validation].to_numpy()
            cluster_auc = roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else np.nan

            cluster_fold_results.append({
                "model": name, "fold": k, "cluster": cluster_id,
                "n_train": int(cluster_train.sum()), "n_val": int(cluster_validation.sum()),
                "accuracy": accuracy_score(y_val, pred),
                "f1": f1_score(y_val, pred, zero_division=0),
                "roc_auc": cluster_auc,
            })
            cluster_aucs.append(cluster_auc)
            cluster_ns.append(len(y_val))
            best_params_rows.append({
                "model": name, "fold": k, "cluster": cluster_id,
                "n_train": int(cluster_train.sum()), "best_params": search.best_params_,
            })

            for pid, yt, p, c in zip(patient_id[cluster_validation], y_val, proba, pred):
                oof_rows.append({
                    "patient_id": int(pid), "fold": int(k), "cluster": int(cluster_id),
                    "y_true": int(yt), "model": name, "proba": float(p), "pred": int(c),
                })

            pooled_y_true.append(y_val)
            pooled_proba.append(proba)
            pooled_pred.append(pred)

        # Population-level pooled result for this outer fold: every cluster's
        # validation predictions are combined before scoring, not averaged,
        # so the pooled metric reflects the full validation fold at once.
        y_true_pooled = np.concatenate(pooled_y_true)
        proba_pooled = np.concatenate(pooled_proba)
        pred_pooled = np.concatenate(pooled_pred)

        # Sample-size-weighted average of each cluster's own AUC -- an
        # alternative to pooling probabilities that avoids mixing multiple
        # independently calibrated models (Section 13).
        aucs_arr = np.array(cluster_aucs, dtype=float)
        ns_arr = np.array(cluster_ns, dtype=float)
        valid = ~np.isnan(aucs_arr)
        roc_auc_weighted = float(np.average(aucs_arr[valid], weights=ns_arr[valid])) if valid.any() else np.nan

        fold_results.append({
            "model": name,
            "fold": k,
            "accuracy": accuracy_score(y_true_pooled, pred_pooled),
            "f1": f1_score(y_true_pooled, pred_pooled, zero_division=0),
            "roc_auc": roc_auc_score(y_true_pooled, proba_pooled),
            "roc_auc_weighted_clusters": roc_auc_weighted,
        })

fold_results_df = pd.DataFrame(fold_results)
cluster_fold_results_df = pd.DataFrame(cluster_fold_results)
cluster_choice_df = pd.DataFrame(cluster_choice)
best_params_df = pd.DataFrame(best_params_rows)
print()
print("Pooled nested cross-validation complete:", len(fold_results_df), "model x fold rows")
print("Cluster-specific nested cross-validation complete:", len(cluster_fold_results_df), "model x fold x cluster rows")

fold 0: selected k=2, training cluster sizes [147, 90]


fold 1: selected k=2, training cluster sizes [141, 96]


fold 2: selected k=2, training cluster sizes [127, 111]


fold 3: selected k=2, training cluster sizes [100, 138]


fold 4: selected k=2, training cluster sizes [104, 134]



Pooled nested cross-validation complete: 10 model x fold rows
Cluster-specific nested cross-validation complete: 20 model x fold x cluster rows


## 12. Fold-level results

The pooled, population-level fold results are the primary output of this notebook and are saved for comparison with Arm A and Arm B. The cluster-specific fold-level results are a secondary, descriptive breakdown, and the per-fold cluster choice (`k_used`, whether that fold fell back to the global model, and the resulting cluster sizes) is reported separately, since it is itself a result of applying a data-driven method to five different training folds rather than a fixed design choice.

In [15]:
fold_results_df.to_csv(os.path.join(RESULTS_DIR, "armC_fold_results.csv"), index=False)
print("Saved armC_fold_results.csv (pooled, primary)")
fold_results_df.round(3)

Saved armC_fold_results.csv (pooled, primary)


,model,fold,accuracy,f1,roc_auc,roc_auc_weighted_clusters
0,logreg,0,0.917,0.912,0.936,0.761
1,rf,0,0.883,0.881,0.953,0.831
2,logreg,1,0.817,0.792,0.888,0.873
3,rf,1,0.750,0.727,0.857,0.880
4,logreg,2,0.780,0.745,0.850,0.769
5,rf,2,0.746,0.717,0.823,0.737
6,logreg,3,0.797,0.760,0.888,0.794
7,rf,3,0.763,0.731,0.869,0.725
8,logreg,4,0.763,0.750,0.841,0.863
9,rf,4,0.864,0.833,0.934,0.885


In [16]:
cluster_choice_df.to_csv(os.path.join(RESULTS_DIR, "armC_cluster_choice.csv"), index=False)
print("Saved armC_cluster_choice.csv")
cluster_choice_df

Saved armC_cluster_choice.csv


,fold,k_used,fallback_to_global,fallback_reason,train_cluster_sizes,val_cluster_sizes
0,0,2,False,None,"[147, 90]","[29, 31]"
1,1,2,False,None,"[141, 96]","[34, 26]"
2,2,2,False,None,"[127, 111]","[35, 24]"
3,3,2,False,None,"[100, 138]","[23, 36]"
4,4,2,False,None,"[104, 134]","[20, 39]"


In [17]:
cluster_fold_results_display = cluster_fold_results_df.round(3)
cluster_fold_results_display

,model,fold,cluster,n_train,n_val,accuracy,f1,roc_auc
0,logreg,0,0,147,29,0.931,0.000,0.852
1,logreg,0,1,90,31,0.903,0.945,0.677
2,rf,0,0,147,29,0.897,0.000,0.852
3,rf,0,1,90,31,0.871,0.929,0.812
4,logreg,1,0,141,34,0.882,0.778,0.905
5,logreg,1,1,96,26,0.731,0.800,0.830
6,rf,1,0,141,34,0.735,0.400,0.854
7,rf,1,1,96,26,0.769,0.850,0.915
8,logreg,2,0,127,35,0.771,0.200,0.782
9,logreg,2,1,111,24,0.792,0.878,0.750


In [18]:
best_params_df.to_csv(os.path.join(DIAGNOSTICS_DIR, "armC_best_params.csv"), index=False)
print("Saved armC_best_params.csv")
best_params_df.head(10)

Saved armC_best_params.csv


,model,fold,cluster,n_train,best_params
0,logreg,0,0,147,{'clf__C': 0.1}
1,logreg,0,1,90,{'clf__C': 10}
2,rf,0,0,147,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
3,rf,0,1,90,"{'clf__max_depth': None, 'clf__n_estimators': ..."
4,logreg,1,0,141,{'clf__C': 10}
5,logreg,1,1,96,{'clf__C': 1}
6,rf,1,0,141,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
7,rf,1,1,96,"{'clf__max_depth': 5, 'clf__n_estimators': 400}"
8,logreg,2,0,127,{'clf__C': 0.1}
9,logreg,2,1,111,{'clf__C': 0.1}


## 13. Overall performance summary

Mean and standard deviation across the 5 outer folds are reported for each model, using the pooled population-level fold results, so that Arm C's primary comparison with Arm A and Arm B rests on the same kind of summary statistic in every notebook. The corresponding cluster-specific summary is reported separately below as a secondary, descriptive result.

In [19]:
results = (
    fold_results_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
results = results.reset_index()

for _, row in results.iterrows():
    print(f"{row['model']:7s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

results.to_csv(os.path.join(RESULTS_DIR, "armC_results.csv"), index=False)
print("Saved armC_results.csv (pooled, primary)")
results.set_index("model").round(3)

logreg  | ACC 0.814 +/- 0.061 | F1 0.792 +/- 0.070 | AUC 0.881 +/- 0.038
rf      | ACC 0.801 +/- 0.067 | F1 0.778 +/- 0.075 | AUC 0.887 +/- 0.054
Saved armC_results.csv (pooled, primary)


,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.814,0.061,0.792,0.070,0.881,0.038
rf,0.801,0.067,0.778,0.075,0.887,0.054


In [20]:
cluster_results = (
    cluster_fold_results_df
    .groupby(["cluster", "model"])[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
cluster_results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
cluster_results = cluster_results.reset_index()

cluster_results.to_csv(os.path.join(RESULTS_DIR, "armC_cluster_specific_results.csv"), index=False)
print("Saved armC_cluster_specific_results.csv (secondary, descriptive)")
print("Note: cluster ids are only comparable within the same fold, not across folds,")
print("since k-prototypes labels clusters independently each time it is fit; this table")
print("aggregates by cluster id anyway, as a rough per-slot summary, not by matched identity.")
cluster_results.round(3)

Saved armC_cluster_specific_results.csv (secondary, descriptive)
Note: cluster ids are only comparable within the same fold, not across folds,
since k-prototypes labels clusters independently each time it is fit; this table
aggregates by cluster id anyway, as a rough per-slot summary, not by matched identity.


,cluster,model,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
0,0,logreg,0.862,0.063,0.566,0.436,0.876,0.082
1,0,rf,0.817,0.100,0.508,0.416,0.826,0.103
2,1,logreg,0.779,0.080,0.591,0.408,0.763,0.057
3,1,rf,0.803,0.071,0.635,0.394,0.809,0.075


**Subgroup-weighted AUC and the calibration-mixing caveat.** Arm C's pooled ROC-AUC above ranks every validation patient in a fold together, using probabilities from as many independently fitted models as that fold has clusters (1-4). Ranking across a cluster boundary partly reflects how those models' probability scales happen to align with each other, not only how well any one of them discriminates within its own cluster -- the same caveat Arm B documents in its Section 13. Accuracy and F1 do not have this problem, since the 0.5 threshold is applied within each cluster before pooling. As a check, each cluster's own AUC (already in `armC_cluster_specific_results.csv`) is combined into a sample-size-weighted average instead of by pooling probabilities, reported below alongside the pooled figure. **Any cross-arm AUC comparison that includes Arm B or Arm C therefore carries a calibration-mixing component that Arm A's pooled AUC does not.**

In [21]:
pooled_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_pooled_mean", "std": "rocauc_pooled_std"})
)
weighted_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc_weighted_clusters"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_weighted_mean", "std": "rocauc_weighted_std"})
)
auc_comparison = pooled_auc_by_model.join(weighted_auc_by_model).round(3)
auc_comparison.to_csv(os.path.join(DIAGNOSTICS_DIR, "armC_auc_pooled_vs_weighted.csv"))
print("Saved armC_auc_pooled_vs_weighted.csv")
auc_comparison

Saved armC_auc_pooled_vs_weighted.csv


,rocauc_pooled_mean,rocauc_pooled_std,rocauc_weighted_mean,rocauc_weighted_std
model,,,,
logreg,0.881,0.038,0.812,0.052
rf,0.887,0.054,0.812,0.077


## 14. Out-of-fold predictions

For every patient, the out-of-fold prediction from the one outer fold in which they were held out is retained, together with the patient identifier, fold id, cluster id (within that fold), true label, model name, predicted probability, and predicted class. This matches Arm A's and Arm B's out-of-fold prediction structure (`patient_id`, `fold`, `y_true`, `model`, `proba`, `pred`) with `cluster` added, so Arm A, Arm B, and Arm C can be compared using the same patient identifiers, fold assignment, and column layout. Because every validation patient is assigned to exactly one cluster in the fold that holds them out, each patient receives exactly one out-of-fold prediction per model.

In [22]:
oof_df = pd.DataFrame(oof_rows)

# Verify exactly one OOF prediction per patient per model, and full coverage of all patients.
counts_per_model = oof_df.groupby("model")["patient_id"].nunique()
assert (counts_per_model == len(df)).all(), "Every patient must receive exactly one OOF prediction per model."
assert oof_df.groupby(["model", "patient_id"]).size().max() == 1, "Duplicate OOF prediction detected for a patient."

oof_df.to_csv(os.path.join(PREDICTIONS_DIR, "armC_predictions.csv"), index=False)
print("Saved armC_predictions.csv:", oof_df.shape)
oof_df.head()

Saved armC_predictions.csv: (594, 7)


,patient_id,fold,cluster,y_true,model,proba,pred
0,10,0,0,0,logreg,0.394257,0
1,17,0,0,0,logreg,0.214062,0
2,19,0,0,0,logreg,0.139182,0
3,21,0,0,0,logreg,0.133127,0
4,25,0,0,0,logreg,0.069799,0


## 15. Interpretation / notes

Section 6's whole-dataset descriptive clustering is characterised properly below, rather than described loosely as "close to evenly sized" -- an earlier, less informative draft of this section did so and missed the actual finding.

In [23]:
cluster_characterization = pd.DataFrame({
    "n": desc.groupby("cluster").size(),
    "disease_rate": desc.groupby("cluster")["disease"].mean(),
    "pct_male": desc.groupby("cluster")["sex"].apply(lambda s: (s == "male").mean() * 100),
    "mean_age": X.groupby(desc_labels)["age"].mean(),
    "mean_thalach": X.groupby(desc_labels)["thalach"].mean(),
    "pct_exang": X.groupby(desc_labels)["exang"].mean() * 100,
    "mean_oldpeak": X.groupby(desc_labels)["oldpeak"].mean(),
    "mean_ca": X.groupby(desc_labels)["ca"].mean(),
    "pct_cp_asymptomatic": X.groupby(desc_labels)["cp"].apply(lambda s: (s == 4.0).mean() * 100),
}).round(2)
cluster_characterization.to_csv(os.path.join(RESULTS_DIR, "armC_cluster_characterization.csv"))
print("Saved armC_cluster_characterization.csv")
print(f"Adjusted Rand index vs sex: {adjusted_rand_score(df['sex'], desc_labels):.3f}")
cluster_characterization

Saved armC_cluster_characterization.csv
Adjusted Rand index vs sex: -0.002


,n,disease_rate,pct_male,mean_age,mean_thalach,pct_exang,mean_oldpeak,mean_ca,pct_cp_asymptomatic
0,103,0.83,77.67,58.60,131.06,66.02,1.95,1.16,75.73
1,151,0.21,59.60,50.95,162.37,9.93,0.44,0.25,31.79
2,43,0.47,72.09,57.44,149.16,32.56,1.08,1.02,37.21


**The actual finding.** k-prototypes recovers a **disease-severity axis**, not a clinical subgroup structure, and it has essentially no relationship to sex (adjusted Rand index vs sex = -0.002; see table above). The whole-dataset descriptive fit (Section 6, corrected scaling) selects k=3, not 2: cluster 0 (n=103) is the high-severity end -- 83% disease rate, mean age 58.6, mean max heart rate only 131, 66% report exercise angina, mean ST depression 1.95, mean vessel count 1.16, 76% asymptomatic chest pain; cluster 1 (n=151) is the low-severity end -- 21% disease rate, younger (mean age 51), higher max heart rate (162), only 10% exercise angina, mean ST depression 0.44; cluster 2 (n=43) sits in between on every one of these features (47% disease rate). Every severity-related clinical feature separates the same way, in the same order, across all three clusters -- this is the direct answer to RQ3. The data-driven method did not recover the clinical stratification variable (sex, ARI ≈ 0), and it did not find a competing patient typology either: it found the same severity gradient the classifier was already going to learn from the raw features, just cut into three ordered bins.

The per-fold leakage-safe evaluation (Sections 10-14) settles on k=2 every fold (`armC_cluster_choice.csv`), but with markedly *uneven* training splits that flip which side is larger from fold to fold (147/90, 141/96, 127/111, 100/138, 104/134) -- a direct consequence of cutting along a severity gradient rather than an even population split: wherever an outer fold's boundary falls, one side of the severity axis simply contains more of that fold's patients than the other. This also explains the cluster-specific instability in `armC_cluster_specific_results.csv`: **both** clusters now show large F1 standard deviations (0.44 and 0.41), not just one -- when a cluster is itself drawn along the outcome axis, its own internal class balance becomes more fold-dependent, reproducing the same regularisation-and-threshold instability documented for Arm B's female subgroup (Section 14 there), just distributed across both clusters instead of concentrated in one subgroup.

This is also why Arm C cannot outperform Arm A by much even where it does reasonably well: because the clusters align so closely with the outcome, Arm C partitions the data roughly along the decision boundary and then asks each partition to find a decision boundary within an already-separated, comparatively homogeneous slice -- there is little separable signal left inside each cluster once the between-cluster separation has done most of the discriminating work. The relevant condition from the cluster-then-predict literature is that this strategy can only improve outcome homogeneity when covariate-space structure corresponds to outcome-space structure; here it corresponds so closely that the correspondence itself, rather than any genuine additional clinical structure, is most of what Arm C's clusters contain.

The calibration-mixing caveat above is not a minor footnote here: pooled and cluster-weighted AUC diverge by as much as 17 points within a single fold (fold 0 logreg: 0.936 pooled vs 0.761 weighted; see `armC_fold_results.csv` and `armC_auc_pooled_vs_weighted.csv`), a far larger gap than the corresponding one in Arm B. Because Arm C's clusters are more unevenly split and more different from each other than Arm B's male/female subgroups, the two per-fold models' probability scales align with each other only loosely, and a meaningful share of the pooled AUC's apparent discriminative power is really just two clusters' very different base rates sitting far apart on the same 0-1 scale -- ranking a near-certain-positive cluster above a near-certain-negative one is easy regardless of how well either model discriminates internally.

For comparability: the same `fold_id.csv` partition (integrity-checked via `run_manifest.json`, Section 8), feature grouping, and classifiers/grids used here match Arm A and Arm B unchanged, so `armC_results.csv` is directly comparable to `armA_results.csv` and `armB_results.csv` on the same terms -- checked for statistical distinguishability in `cross_arm_comparison.ipynb`. `passthrough` for the classifiers (not the clustering step, which now standardises it -- Section 6) is `["sex", "fbs", "exang", "ca"]`, identical to Arm A; Arm B's `["fbs", "exang", "ca"]` is the one deliberate exception, documented in Arm B's Section 5.